# 1 - Mise à jour de la base de données

Ce notebook illustre les fonctionnalités des modules `updater` et `deleter` permettant de mettre à jour et de supprimer des données dans une base DuckLake. Les connexions sont créées via `DuckLakeConnector` et passées aux classes `DatabaseUpdaterV2` et `DatabaseDeleterV2`.

## 0 - Importation des modules

In [1]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Ajout du chemin vers le package parent
sys.path.append('..')

# Importation des modules ad hoc
from dashboard_template_database.builders.connector import DuckLakeConnector
from dashboard_template_database.builders.tables import DuckLakeTablesBuilder
from dashboard_template_database.operations.updater import DatabaseUpdaterV2
from dashboard_template_database.operations.deleter import DatabaseDeleterV2

# Paramètres globaux
CATEGORICAL_THRESHOLD: int = 10
PRIMARY_KEYS: list = ['indicator', 'country', 'kind', 'model', 'training', 'week', 'horizon', 'date']
CATALOG_PATH: str = os.path.join('../outputs', 'database_update_demo.ducklake')
DATA_PATH: str    = os.path.join('../outputs', 'database_update_demo_data/')

## 1 - Construction de la base de données initiale

Construction d'un jeu de données contrôlé (~60 lignes) conçu pour illustrer les scénarios d'apparition et de disparition de tables de dimension.

**État initial visé :**
- `source` : 15 valeurs uniques > seuil=10 → **non-catégorielle**, pas de `dim_source`
- `model` : 3 valeurs uniques ≤ seuil=10 → **catégorielle**, `dim_model` créée

In [2]:
# Initialisation du générateur aléatoire pour la reproductibilité
np.random.seed(0)

# Définition des modalités
indicators_init = ['temperature', 'humidity', 'pressure', 'wind_speed']
countries_init  = ['France', 'Germany', 'Italy', 'Spain', 'Belgium']
kinds_init      = ['forecast', 'observation']
models_init     = ['model_A', 'model_B', 'model_C']
trainings_init  = ['train_v1', 'train_v2']
horizons_init   = [1, 7, 14, 30]
weeks_init      = list(range(1, 5))

# Colonne source : 15 valeurs uniques → non-catégorielle (15 > CATEGORICAL_THRESHOLD=10)
sources_init = [f'source_{i:02d}' for i in range(1, 16)]

# Labels en français
labels_init: dict = {
    'indicator':     'Indicateur',
    'country':       'Pays',
    'kind':          'Type',
    'model':         'Modèle',
    'training':      'Entraînement',
    'week':          'Semaine',
    'horizon':       'Horizon',
    'date':          'Date',
    'value':         'Valeur',
    'lower_bound':   'Borne inférieure',
    'upper_bound':   'Borne supérieure',
    'quality_score': 'Score de qualité',
    'source':        'Source',
    'notes':         'Notes',
}

# Génération de 80 lignes (les doublons sur la clé primaire seront supprimés)
start_date_init = datetime(2024, 1, 1)
rows_init = []
for i in range(80):
    date = start_date_init + timedelta(days=i % 15)
    rows_init.append({
        'indicator':     np.random.choice(indicators_init),
        'country':       np.random.choice(countries_init),
        'kind':          np.random.choice(kinds_init),
        'model':         np.random.choice(models_init),
        'training':      np.random.choice(trainings_init),
        'week':          weeks_init[i % len(weeks_init)],
        'horizon':       horizons_init[i % len(horizons_init)],
        'date':          date,
        'value':         np.random.uniform(10, 100),
        'lower_bound':   None if np.random.random() > 0.7 else np.random.uniform(5, 50),
        'upper_bound':   None if np.random.random() > 0.7 else np.random.uniform(50, 150),
        'quality_score': np.random.uniform(0, 1),
        'source':        np.random.choice(sources_init),
        'notes':         np.random.choice(['OK', 'Warning', None], p=[0.7, 0.2, 0.1]),
    })

# Création et dédoublonnage du DataFrame sur la clé primaire composite
df_demo = pd.DataFrame(rows_init)
df_demo['date'] = pd.to_datetime(df_demo['date'])
df_demo = df_demo.drop_duplicates(subset=PRIMARY_KEYS, keep='first').reset_index(drop=True)

# Vérification des contraintes de cardinalité
assert df_demo['source'].nunique() > CATEGORICAL_THRESHOLD, (
    f"source doit avoir > {CATEGORICAL_THRESHOLD} valeurs uniques "
    f"(actuel : {df_demo['source'].nunique()})"
)
assert df_demo['model'].nunique() <= CATEGORICAL_THRESHOLD, (
    f"model doit avoir ≤ {CATEGORICAL_THRESHOLD} valeurs uniques "
    f"(actuel : {df_demo['model'].nunique()})"
)

# Affichage
print(f"Lignes après dédoublonnage   : {len(df_demo)}")
print(f"source — valeurs uniques     : {df_demo['source'].nunique()} (non-catégoriel)")
print(f"model  — valeurs uniques     : {df_demo['model'].nunique()}  (catégoriel)")
df_demo.head()

Lignes après dédoublonnage   : 80
source — valeurs uniques     : 14 (non-catégoriel)
model  — valeurs uniques     : 3  (catégoriel)


,indicator,country,kind,model,training,week,horizon,date,value,lower_bound,upper_bound,quality_score,source,notes
0,temperature,France,observation,model_B,train_v2,1,1,2024-01-01,68.130470,45.129785,NaN,0.383442,source_07,Warning
1,pressure,France,observation,model_B,train_v2,2,7,2024-01-02,16.393245,5.909828,NaN,0.778157,source_01,None
2,temperature,Germany,observation,model_B,train_v2,3,14,2024-01-03,71.099158,NaN,103.737323,0.758616,source_11,OK
3,wind_speed,Spain,forecast,model_A,train_v1,4,30,2024-01-04,22.169636,11.735369,88.648898,0.902598,source_11,OK
4,temperature,Germany,observation,model_B,train_v2,1,1,2024-01-05,70.009004,14.467215,81.542835,0.363711,source_10,OK


In [3]:
# Suppression du catalogue et des données existants pour garantir un état initial propre
for suffix in ['', '.wal']:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)
    print(f"Répertoire supprimé : {DATA_PATH}")

# Création de la connexion DuckLake et construction du schéma initial
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
builder = DuckLakeTablesBuilder(
    df=df_demo,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=PRIMARY_KEYS,
    connection=conn,
)
builder.build_schema(column_labels=labels_init)
conn.close()

print(f"\nBase de données créée : {CATALOG_PATH}")

2026-04-03 15:52:24,572 - INFO - DuckLake extension loaded
2026-04-03 15:52:24,617 - INFO - DuckLake catalog attached : '../outputs\database_update_demo.ducklake' (alias=db, read_only=False)
2026-04-03 15:52:24,639 - INFO - Activated scheme : db.main
2026-04-03 15:52:24,642 - INFO - Primary keys validated successfully: ['indicator', 'country', 'kind', 'model', 'training', 'week', 'horizon', 'date']
2026-04-03 15:52:24,647 - INFO - Successfully extracted meta-data from column 'indicator'
2026-04-03 15:52:24,648 - INFO - The column 'indicator' is of type 'object' and the number of modalities 4 satisfies the categorical threshold criteria 10
2026-04-03 15:52:24,649 - INFO - Successfully extracted meta-data from column 'country'
2026-04-03 15:52:24,651 - INFO - The column 'country' is of type 'object' and the number of modalities 5 satisfies the categorical threshold criteria 10
2026-04-03 15:52:24,652 - INFO - Successfully extracted meta-data from column 'kind'
2026-04-03 15:52:24,653 - I


Base de données créée : ../outputs\database_update_demo.ducklake


In [4]:
# Vérification de l'état initial : statut catégoriel dans les métadonnées
conn_check = DuckLakeConnector(CATALOG_PATH, DATA_PATH, read_only=True).connect()

# Affichage des méta-données
print("=== État initial — Métadonnées ===")
display(conn_check.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

# Affichage des tables de dimension
print("=== État initial — Tables de dimension ===")
display(conn_check.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

# Affichage du nombre de lignes dans la table des faits
print(f"Lignes dans fact_table : {conn_check.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")
conn_check.close()

2026-04-03 15:52:26,014 - INFO - DuckLake extension loaded
2026-04-03 15:52:26,062 - INFO - DuckLake catalog attached : '../outputs\database_update_demo.ducklake' (alias=db, read_only=True)
2026-04-03 15:52:26,094 - INFO - Activated scheme : db.main


=== État initial — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,model,Modèle,True
4,notes,Notes,True
5,training,Entraînement,True
6,date,Date,False
7,horizon,Horizon,False
8,lower_bound,Borne inférieure,False
9,quality_score,Score de qualité,False


=== État initial — Tables de dimension ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_model
4,dim_notes
5,dim_training


Lignes dans fact_table : 80


## 2 - Mise à jour de la base de données (`DatabaseUpdaterV2`)

Trois scénarios sont illustrés :

1. **Scénario 2.1** : mise à jour neutre (valeurs numériques seulement) — aucun changement de tables de dimension
2. **Scénario 2.2** : l'upsert réduit les modalités de `source` à 5 valeurs (≤ seuil=10) → `dim_source` **créée**
3. **Scénario 2.3** : l'upsert introduit 12 valeurs uniques pour `model` (> seuil=10) → `dim_model` **supprimée**

In [5]:
# Création de la connexion DuckLake et initialisation de l'updater
# La connexion est partagée avec le DatabaseDeleterV2 en Section 3
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
updater = DatabaseUpdaterV2(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    enable_validation=True,
)

print(f"Lignes dans fact_table : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

2026-04-03 15:52:26,873 - INFO - DuckLake extension loaded
2026-04-03 15:52:26,929 - INFO - DuckLake catalog attached : '../outputs\database_update_demo.ducklake' (alias=db, read_only=False)
2026-04-03 15:52:26,983 - INFO - Activated scheme : db.main


Lignes dans fact_table : 80


### Scénario 2.1 — Mise à jour sans changement de tables de dimension

Mise à jour de quelques lignes existantes en modifiant uniquement les colonnes numériques (`value`, `quality_score`, `lower_bound`, `upper_bound`) et textuelles non-catégorielles (`notes`).

Aucune colonne catégorielle ni la colonne `source` ne sont modifiées, donc aucune table de dimension n'est créée ni supprimée.

> **Note technique** : `fact_table` stocke des IDs numériques pour les colonnes catégorielles. La reconstruction du `update_df` nécessite une jointure avec chaque table de dimension pour retrouver les labels humains attendus par `_prepare_dataframe_for_fact_table`.

In [6]:
# Reconstruction des labels pour 3 lignes existantes via jointures avec les tables de dimension
sample_21 = conn.execute("""
    SELECT
        i.label  AS indicator,
        c.label  AS country,
        k.label  AS kind,
        m.label  AS model,
        t.label  AS training,
        f.week,
        f.horizon,
        f.date,
        f.source
    FROM fact_table f
    JOIN dim_indicator i ON f.indicator = i.value
    JOIN dim_country   c ON f.country   = c.value
    JOIN dim_kind      k ON f.kind      = k.value
    JOIN dim_model     m ON f.model     = m.value
    JOIN dim_training  t ON f.training  = t.value
    LIMIT 3
""").fetchdf()

# Modification des colonnes numériques uniquement
update_21 = sample_21.copy()
update_21['value']         = [99.0, 88.0, 77.0]
update_21['quality_score'] = [0.95, 0.85, 0.75]
update_21['lower_bound']   = [5.0, 4.0, 3.0]
update_21['upper_bound']   = [199.0, 180.0, 155.0]
update_21['notes']         = ['OK', 'OK', 'Warning']

print("DataFrame de mise à jour (scénario 2.1) :")
display(update_21)

DataFrame de mise à jour (scénario 2.1) :


,indicator,country,kind,model,training,week,horizon,date,source,value,quality_score,lower_bound,upper_bound,notes
0,temperature,France,observation,model_B,train_v2,1,1,2024-01-01,source_07,99.0,0.95,5.0,199.0,OK
1,pressure,France,observation,model_B,train_v2,2,7,2024-01-02,source_01,88.0,0.85,4.0,180.0,OK
2,temperature,Germany,observation,model_B,train_v2,3,14,2024-01-03,source_11,77.0,0.75,3.0,155.0,Warning


In [7]:
# Vérification de l'état AVANT la mise à jour 2.1
print("=== AVANT la mise à jour 2.1 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== AVANT la mise à jour 2.1 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== AVANT la mise à jour 2.1 — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,model,Modèle,True
4,notes,Notes,True
5,training,Entraînement,True
6,date,Date,False
7,horizon,Horizon,False
8,lower_bound,Borne inférieure,False
9,quality_score,Score de qualité,False


=== AVANT la mise à jour 2.1 — Tables de dimension ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_model
4,dim_notes
5,dim_training


In [8]:
# Exécution de la mise à jour 2.1
success_21 = updater.update_database(
    update_df=update_21,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.1 : {'✓ succès' if success_21 else '✗ échec'}")

2026-04-03 15:52:28,637 - INFO - Validating preconditions for operation: update
2026-04-03 15:52:28,661 - INFO - Starting database update with 3 rows (transaction: True)
2026-04-03 15:52:28,682 - INFO - Started batch tx_1_1775224348: Database update with validation and rollback
2026-04-03 15:52:28,685 - INFO - Created savepoint 'before_major_updates' in batch tx_1_1775224348
2026-04-03 15:52:28,688 - INFO - Added operation to tx_1_1775224348: metadata_update - Update metadata table
2026-04-03 15:52:28,751 - INFO - Type conflict resolution for week: int64 -> int64
2026-04-03 15:52:28,813 - INFO - Type conflict resolution for horizon: int64 -> int64
2026-04-03 15:52:28,869 - INFO - Type conflict resolution for date: datetime64[ns] -> datetime64[ns]
2026-04-03 15:52:28,873 - INFO - Executed operation 0 in tx_1_1775224348: Update metadata table
2026-04-03 15:52:28,876 - INFO - Added operation to tx_1_1775224348: fact_update_direct - Update fact table (direct)
2026-04-03 15:52:29,585 - INFO

Mise à jour 2.1 : ✓ succès


In [9]:
# Vérification de l'état APRÈS la mise à jour 2.1
# Résultat attendu : métadonnées et tables de dimension inchangées
print("=== APRÈS la mise à jour 2.1 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.1 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== APRÈS la mise à jour 2.1 — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,model,Modèle,True
4,notes,Notes,True
5,source,Source,True
6,training,Entraînement,True
7,date,Date,False
8,horizon,Horizon,False
9,lower_bound,Borne inférieure,False


=== APRÈS la mise à jour 2.1 — Tables de dimension ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_model
4,dim_notes
5,dim_source
6,dim_training


### Scénario 2.2 — Upsert créant une table de dimension (`source` → catégorielle)

L'intégralité des lignes de `fact_table` est mise à jour avec 5 nouvelles valeurs pour `source` au lieu de 15. Comme 5 ≤ seuil=10, `_update_dimensions_safe()` détecte que la colonne non-catégorielle `source` franchit désormais le seuil et appelle `dimension_mgr.convert_to_categorical('source', values)`, ce qui :
1. Crée `dim_source` avec 5 entrées
2. Met à jour `metadata.source.is_categorical` à `True`
3. Convertit les valeurs de `fact_table.source` en IDs numériques

> **Point critique** : la mise à jour doit couvrir **toutes** les lignes. Si certaines lignes ne sont pas incluses dans `update_df`, leurs valeurs `source` restent inchangées (anciens labels) mais `convert_to_categorical` tentera de les convertir en IDs via `_convert_fact_table_dimension_mapping`. Toute valeur absente de `dim_source` serait mise à `NULL`.

In [10]:
# Reconstruction de TOUTES les lignes avec les labels des colonnes catégorielles
# Nécessaire car fact_table stocke des IDs numériques pour les colonnes catégorielles
all_rows_22 = conn.execute("""
    SELECT
        i.label  AS indicator,
        c.label  AS country,
        k.label  AS kind,
        m.label  AS model,
        t.label  AS training,
        f.week,
        f.horizon,
        f.date,
        f.value,
        f.lower_bound,
        f.upper_bound,
        f.quality_score,
        f.source,
        n.label  AS notes
    FROM fact_table f
    JOIN dim_indicator i ON f.indicator = i.value
    JOIN dim_country   c ON f.country   = c.value
    JOIN dim_kind      k ON f.kind      = k.value
    JOIN dim_model     m ON f.model     = m.value
    JOIN dim_training  t ON f.training  = t.value
    LEFT JOIN dim_notes n ON f.notes    = n.value
""").fetchdf()

# Remplacement de source par 5 nouvelles valeurs (couverture uniforme sur toutes les lignes)
sources_new = ['alpha', 'beta', 'gamma', 'delta', 'epsilon']
all_rows_22['source'] = [sources_new[i % len(sources_new)] for i in range(len(all_rows_22))]

# Affichage
print(f"Lignes dans update_df          : {len(all_rows_22)}")
print(f"Valeurs uniques de source      : {sorted(all_rows_22['source'].unique())}")
print(f"Nombre de valeurs uniques      : {all_rows_22['source'].nunique()} (≤ seuil={CATEGORICAL_THRESHOLD} → conversion attendue)")

Lignes dans update_df          : 80
Valeurs uniques de source      : ['alpha', 'beta', 'delta', 'epsilon', 'gamma']
Nombre de valeurs uniques      : 5 (≤ seuil=10 → conversion attendue)


In [11]:
# Vérification de l'état AVANT la mise à jour 2.2
print("=== AVANT la mise à jour 2.2 ===")
print("Statut de source dans les métadonnées :")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata WHERE name = 'source'"
).fetchdf())

print("Tables de dimension (dim_source absente) :")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== AVANT la mise à jour 2.2 ===
Statut de source dans les métadonnées :


,name,label,is_categorical
0,source,Source,True


Tables de dimension (dim_source absente) :


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_model
4,dim_notes
5,dim_source
6,dim_training


In [12]:
# Exécution de la mise à jour 2.2
success_22 = updater.update_database(
    update_df=all_rows_22,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.2 : {'✓ succès' if success_22 else '✗ échec'}")

2026-04-03 15:52:33,769 - INFO - Validating preconditions for operation: update
2026-04-03 15:52:33,790 - INFO - Starting database update with 80 rows (transaction: True)
2026-04-03 15:52:33,796 - INFO - Started batch tx_2_1775224353: Database update with validation and rollback
2026-04-03 15:52:33,799 - INFO - Created savepoint 'before_major_updates' in batch tx_2_1775224353
2026-04-03 15:52:33,800 - INFO - Added operation to tx_2_1775224353: metadata_update - Update metadata table
2026-04-03 15:52:33,838 - INFO - Type conflict resolution for week: int64 -> int64
2026-04-03 15:52:33,872 - INFO - Type conflict resolution for horizon: int64 -> int64
2026-04-03 15:52:33,906 - INFO - Type conflict resolution for date: datetime64[ns] -> datetime64[ns]
2026-04-03 15:52:33,909 - INFO - Executed operation 0 in tx_2_1775224353: Update metadata table
2026-04-03 15:52:33,912 - INFO - Added operation to tx_2_1775224353: fact_update_direct - Update fact table (direct)
2026-04-03 15:52:34,051 - WAR

Mise à jour 2.2 : ✓ succès


In [13]:
# Vérification de l'état APRÈS la mise à jour 2.2
# Résultat attendu : source is_categorical=True, dim_source créée avec 5 entrées
print("=== APRÈS la mise à jour 2.2 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.2 — Tables de dimension (dim_source créée) ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== APRÈS la mise à jour 2.2 — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,model,Modèle,True
4,notes,Notes,True
5,source,Source,True
6,training,Entraînement,True
7,date,Date,False
8,horizon,Horizon,False
9,lower_bound,Borne inférieure,False


=== APRÈS la mise à jour 2.2 — Tables de dimension (dim_source créée) ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_model
4,dim_notes
5,dim_source
6,dim_training


In [14]:
# Contenu de la nouvelle table dim_source
print("Contenu de dim_source :")
display(conn.execute("SELECT * FROM dim_source ORDER BY value").fetchdf())

# Vérification : fact_table.source contient désormais des IDs numériques
print("\nValeurs distinctes de source dans fact_table (IDs après conversion) :")
display(conn.execute("SELECT DISTINCT source FROM fact_table ORDER BY source").fetchdf())

Contenu de dim_source :


,value,label
0,3,alpha
1,4,epsilon
2,5,delta
3,6,gamma
4,7,beta



Valeurs distinctes de source dans fact_table (IDs après conversion) :


,source
0,3
1,4
2,5
3,6
4,7


### Scénario 2.3 — Upsert supprimant une table de dimension (`model` → non-catégorielle)

Insertion de 12 nouvelles lignes dont la colonne `model` contient 12 valeurs uniques (model_A à model_L). Comme 12 > seuil=10, `_update_dimensions_safe()` détecte que la colonne catégorielle `model` dépasse le seuil et appelle `dimension_mgr.convert_to_non_categorical('model')`, ce qui :
1. Convertit les IDs dans `fact_table.model` en labels VARCHAR
2. Supprime `dim_model`
3. Met à jour `metadata.model.is_categorical` à `False`

> **Note** : après le scénario 2.2, `source` est catégorielle avec `dim_source = {alpha, beta, gamma, delta, epsilon}`. Les nouvelles lignes doivent utiliser un label existant pour `source`.

In [15]:
# Construction de 12 nouvelles lignes, une par valeur de model (model_A à model_L)
# Combinaison de clé primaire fixe pour éviter tout conflit avec les lignes existantes
all_models = [f'model_{chr(65 + i)}' for i in range(12)]  # model_A ... model_L

rows_23 = []
for model in all_models:
    rows_23.append({
        'indicator':     'temperature',
        'country':       'France',
        'kind':          'forecast',
        'model':         model,
        'training':      'train_v1',
        'week':          52,
        'horizon':       30,
        'date':          pd.Timestamp('2025-01-01'),
        'source':        'alpha',    # label existant dans dim_source (catégorielle après 2.2)
        'value':         np.random.uniform(10, 100),
        'lower_bound':   None,
        'upper_bound':   None,
        'quality_score': np.random.uniform(0, 1),
        'notes':         'OK',
    })

update_23 = pd.DataFrame(rows_23)

# Affichage
print(f"Lignes dans update_df          : {len(update_23)}")
print(f"Valeurs uniques de model       : {sorted(update_23['model'].unique())}")
print(f"Nombre de valeurs uniques      : {update_23['model'].nunique()} (> seuil={CATEGORICAL_THRESHOLD} → suppression attendue de dim_model)")

Lignes dans update_df          : 12
Valeurs uniques de model       : ['model_A', 'model_B', 'model_C', 'model_D', 'model_E', 'model_F', 'model_G', 'model_H', 'model_I', 'model_J', 'model_K', 'model_L']
Nombre de valeurs uniques      : 12 (> seuil=10 → suppression attendue de dim_model)


In [16]:
# Vérification de l'état AVANT la mise à jour 2.3
print("=== AVANT la mise à jour 2.3 ===")
print("Statut de model dans les métadonnées :")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata WHERE name = 'model'"
).fetchdf())

print("Contenu actuel de dim_model :")
display(conn.execute("SELECT * FROM dim_model ORDER BY value").fetchdf())

=== AVANT la mise à jour 2.3 ===
Statut de model dans les métadonnées :


,name,label,is_categorical
0,model,Modèle,True


Contenu actuel de dim_model :


,value,label
0,0,model_B
1,1,model_A
2,2,model_C


In [17]:
# Exécution de la mise à jour 2.3
success_23 = updater.update_database(
    update_df=update_23,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep='last',
    use_batch_processing=False,
    use_transaction=True,
)
print(f"Mise à jour 2.3 : {'✓ succès' if success_23 else '✗ échec'}")

2026-04-03 15:52:38,075 - INFO - Validating preconditions for operation: update
2026-04-03 15:52:38,095 - INFO - Starting database update with 12 rows (transaction: True)
2026-04-03 15:52:38,101 - INFO - Started batch tx_3_1775224358: Database update with validation and rollback
2026-04-03 15:52:38,104 - INFO - Created savepoint 'before_major_updates' in batch tx_3_1775224358
2026-04-03 15:52:38,105 - INFO - Added operation to tx_3_1775224358: metadata_update - Update metadata table
2026-04-03 15:52:38,119 - INFO - Executed operation 0 in tx_3_1775224358: Update metadata table
2026-04-03 15:52:38,121 - INFO - Added operation to tx_3_1775224358: fact_update_direct - Update fact table (direct)
2026-04-03 15:52:38,326 - INFO - Added 9 new values to dim_model
2026-04-03 15:52:38,600 - INFO - Fact table (direct): 12 inserted, 0 updated
2026-04-03 15:52:38,602 - INFO - Executed operation 1 in tx_3_1775224358: Update fact table (direct)
2026-04-03 15:52:38,603 - INFO - Added operation to tx_3

Mise à jour 2.3 : ✓ succès


In [18]:
# Vérification de l'état APRÈS la mise à jour 2.3
# Résultat attendu : model is_categorical=False, dim_model supprimée
print("=== APRÈS la mise à jour 2.3 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la mise à jour 2.3 — Tables de dimension (dim_model supprimée) ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== APRÈS la mise à jour 2.3 — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,notes,Notes,True
4,source,Source,True
5,training,Entraînement,True
6,date,Date,False
7,horizon,Horizon,False
8,lower_bound,Borne inférieure,False
9,model,Modèle,False


=== APRÈS la mise à jour 2.3 — Tables de dimension (dim_model supprimée) ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_notes
4,dim_source
5,dim_training


In [19]:
# Vérification : fact_table.model contient désormais des labels VARCHAR (pas des IDs)
print("Valeurs distinctes de model dans fact_table (labels VARCHAR après conversion) :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

Valeurs distinctes de model dans fact_table (labels VARCHAR après conversion) :


,model
0,model_A
1,model_B
2,model_C
3,model_D
4,model_E
5,model_F
6,model_G
7,model_H
8,model_I
9,model_J


## 3 - Suppression de données (`DatabaseDeleterV2`)

Deux scénarios sont illustrés :

1. **Scénario 3.1** : suppression des lignes de la Belgique — aucun changement de tables de dimension (model conserve 12 valeurs uniques > seuil)
2. **Scénario 3.2** : suppression des lignes `model_D` à `model_L` — `model` retrouve 3 valeurs uniques ≤ seuil=10 → `dim_model` **recréée**

In [20]:
# Initialisation du deleter en partageant la connexion ouverte par l'updater
# Les deux classes opèrent sur le même état de la base de données
deleter = DatabaseDeleterV2(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    enable_validation=True,
    auto_cleanup=True,
)

print(f"Lignes dans fact_table avant suppressions : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

Lignes dans fact_table avant suppressions : 92


### Scénario 3.1 — Suppression sans changement de tables de dimension

Suppression de toutes les lignes correspondant à la Belgique.

Après la suppression et le nettoyage (`perform_cleanup=True`), l'entrée 'Belgium' est retirée de `dim_country` (nettoyage des orphelins). Cependant :
- Aucune nouvelle table de dimension n'est créée
- Aucune table de dimension existante n'est supprimée

La colonne `model` conserve ses 12 valeurs uniques (les lignes insérées en 2.3 pour la France avec `model_D` à `model_L` sont préservées).

> **Note** : `country` est catégorielle → le filtre utilise l'ID numérique stocké dans `fact_table`, récupéré depuis `dim_country`.

In [21]:
# Récupération de l'identifiant de Belgium dans dim_country
# country est catégorielle → fact_table stocke des IDs → le filtre doit utiliser l'ID
belgium_id = conn.execute(
    "SELECT value FROM dim_country WHERE label = 'Belgium'"
).fetchone()[0]

print(f"ID de Belgium dans dim_country : {belgium_id!r}")

# Comptage préalable des lignes à supprimer
n_belgium = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE country = '{belgium_id}'"
).fetchone()[0]
print(f"Lignes à supprimer             : {n_belgium}")

ID de Belgium dans dim_country : '3'
Lignes à supprimer             : 16


In [22]:
# Vérification de l'état AVANT la suppression 3.1
print("=== AVANT la suppression 3.1 — dim_country ===")
display(conn.execute("SELECT * FROM dim_country ORDER BY value").fetchdf())

print(f"Valeurs uniques de model : {conn.execute('SELECT COUNT(DISTINCT model) FROM fact_table').fetchone()[0]}")

=== AVANT la suppression 3.1 — dim_country ===


,value,label
0,0,France
1,1,Germany
2,2,Spain
3,3,Belgium
4,4,Italy


Valeurs uniques de model : 12


In [23]:
# Exécution de la suppression 3.1
filters_31 = [('country', '=', belgium_id)]

deleted_31 = deleter.delete_rows(
    filters=filters_31,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_31}")

2026-04-03 15:52:41,780 - INFO - Validating preconditions for operation: delete
2026-04-03 15:52:41,782 - INFO - Starting row deletion (transaction: True, cleanup: True)
2026-04-03 15:52:41,787 - INFO - Started batch tx_1_1775224361: Row deletion with cleanup and validation
2026-04-03 15:52:41,793 - INFO - Added operation to tx_1_1775224361: delete_rows - Delete rows from fact table
2026-04-03 15:52:41,840 - INFO - Successfully deleted 16 rows from fact_table
2026-04-03 15:52:41,841 - INFO - Executed operation 0 in tx_1_1775224361: Delete rows from fact table
2026-04-03 15:52:41,848 - INFO - Created savepoint 'before_cleanup' in batch tx_1_1775224361
2026-04-03 15:52:41,849 - INFO - Added operation to tx_1_1775224361: cleanup_orphaned - Clean up orphaned dimension entries and null columns
2026-04-03 15:52:42,002 - INFO - Removed 1 orphaned entries from dim_country
2026-04-03 15:52:42,174 - INFO - Executed operation 1 in tx_1_1775224361: Clean up orphaned dimension entries and null colu

Lignes supprimées : 16


In [24]:
# Vérification de l'état APRÈS la suppression 3.1
# Résultat attendu : Belgium absent de dim_country, aucune table créée/supprimée
print("=== APRÈS la suppression 3.1 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la suppression 3.1 — Tables de dimension ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== APRÈS la suppression 3.1 — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,notes,Notes,True
4,source,Source,True
5,training,Entraînement,True
6,date,Date,False
7,horizon,Horizon,False
8,lower_bound,Borne inférieure,False
9,model,Modèle,False


=== APRÈS la suppression 3.1 — Tables de dimension ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_notes
4,dim_source
5,dim_training


In [25]:
# Belgium doit avoir disparu de dim_country (entrée orpheline supprimée)
print("Contenu de dim_country après suppression :")
display(conn.execute("SELECT * FROM dim_country ORDER BY value").fetchdf())

# model doit toujours avoir 12 valeurs uniques (France/model_D...model_L préservées)
unique_model_count = conn.execute(
    "SELECT COUNT(DISTINCT model) FROM fact_table"
).fetchone()[0]
print(f"\nValeurs uniques de model après suppression Belgique : {unique_model_count} (> seuil={CATEGORICAL_THRESHOLD} → pas de conversion)")

Contenu de dim_country après suppression :


,value,label
0,0,France
1,1,Germany
2,2,Spain
3,4,Italy



Valeurs uniques de model après suppression Belgique : 12 (> seuil=10 → pas de conversion)


### Scénario 3.2 — Suppression créant une nouvelle table de dimension (`model` → catégorielle)

Suppression des lignes dont le modèle appartient à `{model_D, ..., model_L}`. Après la suppression, seuls `model_A`, `model_B`, `model_C` subsistent (3 valeurs ≤ seuil=10).

Le nettoyage post-suppression (`perform_cleanup=True`) déclenche `_detect_new_categorical_after_deletion()` qui détecte ce franchissement de seuil et appelle `convert_to_categorical('model')` :
1. Crée `dim_model` avec 3 entrées
2. Convertit les labels VARCHAR de `fact_table.model` en IDs numériques
3. Met à jour `metadata.model.is_categorical` à `True`

> **Note** : `model` est non-catégorielle après le scénario 2.3 → `fact_table.model` stocke des labels VARCHAR → le filtre SQL utilise les labels directement.

In [26]:
# Modèles à supprimer (ceux introduits en 2.3 qui dépassent le seuil)
models_to_delete = [f'model_{chr(65 + i)}' for i in range(3, 12)]  # model_D ... model_L
print(f"Modèles à supprimer : {models_to_delete}")

# Comptage préalable des lignes à supprimer
models_sql = ', '.join([f"'{m}'" for m in models_to_delete])
n_to_delete = conn.execute(
    f"SELECT COUNT(*) FROM fact_table WHERE model IN ({models_sql})"
).fetchone()[0]
print(f"Lignes à supprimer : {n_to_delete}")

# Filtre SQL en chaîne de caractères (model est VARCHAR non-catégorielle → labels directs)
filter_32 = f"model IN ({models_sql})"
print(f"Filtre SQL         : {filter_32}")

Modèles à supprimer : ['model_D', 'model_E', 'model_F', 'model_G', 'model_H', 'model_I', 'model_J', 'model_K', 'model_L']
Lignes à supprimer : 9
Filtre SQL         : model IN ('model_D', 'model_E', 'model_F', 'model_G', 'model_H', 'model_I', 'model_J', 'model_K', 'model_L')


In [27]:
# Vérification de l'état AVANT la suppression 3.2
print("=== AVANT la suppression 3.2 ===")
print("Valeurs distinctes de model dans fact_table :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

print(f"\nTotal lignes dans fact_table : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

=== AVANT la suppression 3.2 ===
Valeurs distinctes de model dans fact_table :


,model
0,model_A
1,model_B
2,model_C
3,model_D
4,model_E
5,model_F
6,model_G
7,model_H
8,model_I
9,model_J



Total lignes dans fact_table : 76


In [28]:
# Exécution de la suppression 3.2
# perform_cleanup=True est essentiel pour déclencher _detect_new_categorical_after_deletion()
deleted_32 = deleter.delete_rows(
    filters=filter_32,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {deleted_32}")

2026-04-03 15:52:44,098 - INFO - Validating preconditions for operation: delete
2026-04-03 15:52:44,100 - INFO - Starting row deletion (transaction: True, cleanup: True)
2026-04-03 15:52:44,105 - INFO - Started batch tx_2_1775224364: Row deletion with cleanup and validation
2026-04-03 15:52:44,110 - INFO - Added operation to tx_2_1775224364: delete_rows - Delete rows from fact table
2026-04-03 15:52:44,167 - INFO - Successfully deleted 9 rows from fact_table
2026-04-03 15:52:44,169 - INFO - Executed operation 0 in tx_2_1775224364: Delete rows from fact table
2026-04-03 15:52:44,176 - INFO - Created savepoint 'before_cleanup' in batch tx_2_1775224364
2026-04-03 15:52:44,177 - INFO - Added operation to tx_2_1775224364: cleanup_orphaned - Clean up orphaned dimension entries and null columns
2026-04-03 15:52:44,514 - INFO - Created dimension table dim_model with 3 unique values
2026-04-03 15:52:44,540 - INFO - Updated categorical status for model: True
2026-04-03 15:52:44,601 - INFO - Conv

Lignes supprimées : 9


In [29]:
# Vérification de l'état APRÈS la suppression 3.2
# Résultat attendu : model is_categorical=True, dim_model recréée avec 3 entrées
print("=== APRÈS la suppression 3.2 — Métadonnées ===")
display(conn.execute(
    "SELECT name, label, is_categorical FROM metadata ORDER BY is_categorical DESC, name"
).fetchdf())

print("=== APRÈS la suppression 3.2 — Tables de dimension (dim_model recréée) ===")
display(conn.execute("""
    SELECT table_name FROM information_schema.tables
    WHERE table_name LIKE 'dim_%' ORDER BY table_name
""").fetchdf())

=== APRÈS la suppression 3.2 — Métadonnées ===


,name,label,is_categorical
0,country,Pays,True
1,indicator,Indicateur,True
2,kind,Type,True
3,model,Modèle,True
4,notes,Notes,True
5,source,Source,True
6,training,Entraînement,True
7,date,Date,False
8,horizon,Horizon,False
9,lower_bound,Borne inférieure,False


=== APRÈS la suppression 3.2 — Tables de dimension (dim_model recréée) ===


,table_name
0,dim_country
1,dim_indicator
2,dim_kind
3,dim_model
4,dim_notes
5,dim_source
6,dim_training


In [30]:
# dim_model doit être recréée avec 3 entrées
print("Contenu de dim_model (recréée) :")
display(conn.execute("SELECT * FROM dim_model ORDER BY value").fetchdf())

# fact_table.model contient à nouveau des IDs numériques
print("\nValeurs distinctes de model dans fact_table (IDs numériques après reconversion) :")
display(conn.execute("SELECT DISTINCT model FROM fact_table ORDER BY model").fetchdf())

print(f"\nTotal lignes dans fact_table : {conn.execute('SELECT COUNT(*) FROM fact_table').fetchone()[0]}")

Contenu de dim_model (recréée) :


,value,label
0,0,model_C
1,1,model_B
2,2,model_A



Valeurs distinctes de model dans fact_table (IDs numériques après reconversion) :


,model
0,0
1,1
2,2



Total lignes dans fact_table : 67
